In [0]:
dbutils.widgets.text("catalog", "fk_ce1366")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.read.table(f"{catalog}.bronze.customers")

#casting & data quality
df = (
    df.filter(col("customer_id").isNotNull())
    .withColumn("full_name", trim(initcap(col("full_name"))))
    .withColumn("email", trim(lower(col("email"))))
    .withColumn("phone", trim(regexp_replace(col("phone"), "[^0-9+]", "")))
    .withColumn("city", trim(initcap(col("city"))))
    .withColumn("tier", trim(initcap(col("tier"))))
    .withColumn("signup_date", try_to_date(col("signup_date"),  "yyyy-MM-dd"))
)

df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.customers")


In [0]:
from pyspark.sql.functions import *

target_table = f"{catalog}.silver.customers_scd2"
source_table = f"{catalog}.silver.customers"

# Columns to compare when detecting changes
compare_cols = ["full_name", "email", "phone", "city", "tier", "signup_date"]
changed_cols = " OR ".join([f"s.{c} <=> t.{c} = false" for c in compare_cols])

if not spark.catalog.tableExists(target_table):
    spark.sql(f"""
        CREATE OR REPLACE TABLE {target_table} AS
        SELECT
            customer_id,
            full_name,
            email,
            phone,
            city,
            tier,
            signup_date,
            current_date() AS effective_from,
            CAST(NULL AS DATE) AS effective_to,
            true AS is_current
        FROM {source_table}
    """)
else:
    # Step 1 — Expire previous current rows for customers whose tracked attributes changed
    spark.sql(f"""
        MERGE INTO {target_table} AS t
        USING (
            SELECT s.customer_id
            FROM {source_table} AS s
            JOIN {target_table} AS t
              ON s.customer_id = t.customer_id
            WHERE t.is_current = true
              AND ({changed_cols})
        ) AS c
        ON t.customer_id = c.customer_id AND t.is_current = true
        WHEN MATCHED THEN UPDATE SET
            effective_to = current_date(),
            is_current = false
    """)

    spark.sql(f"""
        INSERT INTO {target_table}
        SELECT
            s.customer_id,
            s.full_name,
            s.email,
            s.phone,
            s.city,
            s.tier,
            s.signup_date,
            current_date() AS effective_from,
            CAST(NULL AS DATE) AS effective_to,
            true AS is_current
        FROM {source_table} AS s
        LEFT JOIN {target_table} AS t
          ON s.customer_id = t.customer_id AND t.is_current = true
        WHERE t.customer_id IS NULL           
           OR ({changed_cols})                
    """)